# 🎓 Capstone Project · "What is this market data telling us?"
### Phase 1 · Data Analysis & Exploration

| | |
|---|---|
| **Your role** | Junior data analyst at *AgriInsight Kenya*, a (fictional) agricultural market-information service |
| **Your client** | The head of analytics, who is deciding whether the company can build price-forecasting models next quarter |
| **Time** | 6–10 hours over 2–3 days |
| **Deliverable** | **This notebook**, completed, run top-to-bottom, readable by a non-programmer |

---

## 📩 The brief

> *Subject: First look at the market-price data*
>
> Hi,
>
> Last month we received four data files from our field teams: weekly market prices, county crop production, market details and monthly weather.
> Nobody has looked at them properly. Before we spend a quarter building forecasting models, I need to know:
>
> 1. **What is in these data**, and **can we trust them**?
> 2. **What patterns** do prices follow: across crops, markets, counties and time?
> 3. **What seems to drive prices** (supply? rain? season? location?), and what only *looks* like it does?
> 4. **Is the data ready for machine learning**, and what exactly would we need to fix or build first?
>
> Please send me a notebook I can read in 20 minutes. I care more about *what you found and how sure you are* than about code.
>
> Thanks: it's yours.

You've now got the whole toolkit. This is the part where you **use** it, with no one holding your hand: only a rubric, a scaffold, and a reference solution you may open **after** you finish.

---

## 📦 Your data

| File | What it is |
|---|---|
| `data/market_prices_raw.csv` | weekly price (KES/kg) and traded volume (kg), by crop, market, date. **Raw.** Real-world damage included. |
| `data/production_by_county.xlsx` | sheet `production` (county × crop × year: area, tonnes; title rows on top!) and sheet `counties` (region, population) |
| `data/markets.json` | per-market type (wholesale/retail), coordinates, opening days, crops traded (nested JSON) |
| `data/weather_monthly.json` | monthly rainfall (mm) and average temperature (°C) per county |

> ⚠️ **Use the *raw* price file.** Finding what's wrong with it is a graded part of the project.
> (A tidy teaching twin, `market_prices_clean.csv`, also lives in `data/` for the lessons. Using it here would skip the detective work, and the learning.)

---

## 📏 The rubric (100 points)

| # | Section | Pts | What "excellent" looks like |
|---|---|---|---|
| 1 | **Dataset overview** | 10 | Plain-language description of every file: rows, columns, period, unit of analysis (*what is one row?*), data dictionary, coverage |
| 2 | **Data-quality assessment & cleaning** | 20 | Every issue **found, counted, explained and fixed** (or deliberately left), with a *decision log*; quality assertions at the end |
| 3 | **Descriptive statistics** | 10 | Centre, spread and shape by crop / market / year; right measure chosen (median vs mean, CV vs std) and interpreted |
| 4 | **Visualisations** | 20 | ≥ 6 charts, each with a *finding as title*, labelled axes with units, the right chart type, no misleading scales |
| 5 | **Correlation analysis** | 15 | Correlations computed *and challenged*: stratified, detrended, lagged where needed; confounding discussed |
| 6 | **Observations & conclusions** | 15 | ≥ 5 findings with numbers; a **hypothesis table** (observation → hypothesis → test → ML implication); honest limitations |
| 7 | **ML-readiness** | 10 | Candidate targets, feature ideas, **leakage** risks, a sensible **train/test split**, and a **naive baseline** to beat |

**Scoring guide for each section:** 
**Excellent (90–100%)** correct, insightful, explained · **Good (70–89%)** correct but descriptive, little interpretation · **Needs work (< 70%)** errors, missing steps, or numbers without meaning.

### ✍️ Writing standards
* Every chart: **title = the finding**, axis labels with **units**, readable ticks.
* Every section ends with a **"So what?"** markdown cell: 2–4 sentences of plain English.
* Numbers in your conclusions come from code (no typing numbers from memory).
* Run **Kernel → Restart & Run All** before you finish. It must run top to bottom without errors.

---

## 🧭 Suggested plan

| Block | Do | Time |
|---|---|---|
| 1 | Load everything, sections 1–2 (overview + audit) | 2 h |
| 2 | Cleaning pipeline + quality gates | 1.5 h |
| 3 | Sections 3–4 (statistics + charts) | 2 h |
| 4 | Section 5 (correlation): *the thinking-heavy part* | 1.5 h |
| 5 | Sections 6–7 (conclusions + ML readiness), polish | 1.5 h |

---

## 🛠️ Setup

In [1]:
import sys; sys.path.insert(0, "..")
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from course_utils import DATA_DIR, first_look

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({"figure.figsize": (10, 4.5), "figure.dpi": 100})
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)

# --- Load the four sources (all RAW) ---------------------------------------------------------
prices_raw = pd.read_csv(DATA_DIR / "market_prices_raw.csv")
production_raw = pd.read_excel(DATA_DIR / "production_by_county.xlsx", sheet_name="production", skiprows=2)
counties = pd.read_excel(DATA_DIR / "production_by_county.xlsx", sheet_name="counties")
weather = pd.read_json(DATA_DIR / "weather_monthly.json")
with open(DATA_DIR / "markets.json") as f:
    markets = pd.json_normalize(json.load(f))
markets.columns = markets.columns.str.replace(".", "_", regex=False)

print("prices_raw    :", prices_raw.shape)
print("production_raw:", production_raw.shape)
print("counties      :", counties.shape)
print("weather       :", weather.shape)
print("markets       :", markets.shape)

prices_raw    : (15093, 9)
production_raw: (204, 5)
counties      : (9, 3)
weather       : (432, 5)
markets       : (10, 8)


---
# 1 · Dataset overview  *(10 pts)*

**Goal:** a reader who has never seen these files should understand them after this section.

**Cover:**
* For **each** of the five tables: rows × columns, what **one row** represents, key columns, units.
* A **data dictionary** (column → meaning → type → unit) for the main price table.
* Coverage: date range, number of crops / markets / counties; which markets sell which crops; are there obvious gaps?

<details><summary>💡 Hint: structure</summary>

Use `first_look()` for each table, then summarise in a small DataFrame you build by hand (`pd.DataFrame({"table": [...], "rows": [...], "one row is": [...]})`).
A `pd.crosstab(market, crop)` heatmap shows coverage at a glance.
</details>

In [2]:
# TODO 1.1  First look at each table
# first_look(prices_raw)

In [3]:
# TODO 1.2  A table describing every source: rows, columns, "one row is…", period, key columns

In [4]:
# TODO 1.3  Data dictionary for the price table (build it as a DataFrame)

In [5]:
# TODO 1.4  Coverage: date range, counts of crops/markets/counties, market × crop heatmap

### 🧾 So what?: *(write 2–4 sentences: what are these data, and what are they good for?)*

✍️ 

---
# 2 · Data-quality assessment & cleaning  *(20 pts)*

**Goal:** find **every** problem, quantify it, decide what to do, fix it, and *prove* it's fixed.

Work through the **seven dimensions** from Lesson 4: *completeness · uniqueness · validity · consistency · plausibility · continuity · integrity (across tables)*.

**Deliverables**
1. An **issues table**: `dimension | issue | rows affected | evidence | decision`.
2. A **cleaning function** `clean_prices(raw)` (and any fixes for the production table).
3. **Quality gates**: `assert` statements proving the cleaned data satisfy your rules.
4. A **decision log** in markdown: for each non-trivial choice, say *why* (e.g. "set to missing rather than dropping because…").

<details><summary>💡 Hint 1, where to look (categories, not answers)</summary>

Every problem below is present in at least one table. Your job is to *find* them and *count* them:
* repeated rows · gaps (and **are the gaps random?**) · inconsistent labels · values that are impossible · values in the wrong *units* ·
  values that are *possible* but wildly out of line with their neighbours · keys that don't match across tables · whether weekly dates are complete.
</details>

<details><summary>💡 Hint 2: detective tools</summary>

`duplicated`, `isna`, `value_counts` on every text column, `describe` (min/max!), `groupby(...).transform("median")` to compare each value with its own crop–market neighbourhood,
`merge(..., indicator=True)` for key mismatches, `date.diff().value_counts()` for date continuity.
</details>

<details><summary>💡 Hint 3: order of operations</summary>

Fix **systematic** problems first (duplicates, spelling, **units**), *then* hunt for random outliers, *then* deal with gaps. If you hunt outliers before fixing units, you'll throw away valid data.
</details>

In [6]:
# TODO 2.1  Audit: completeness, uniqueness, validity, consistency (prices_raw)

In [7]:
# TODO 2.2  Audit: plausibility (outliers vs each series' own neighbourhood), dates, units

In [8]:
# TODO 2.3  Audit: production + counties + markets + weather (keys, gaps, spelling)

In [9]:
# TODO 2.4  The issues table
issues = pd.DataFrame(columns=["dimension", "issue", "rows_affected", "evidence", "decision"])
issues

,dimension,issue,rows_affected,evidence,decision


In [10]:
# TODO 2.5  clean_prices(raw) -> tidy DataFrame  (and production cleanup)
def clean_prices(raw):
    d = raw.copy()
    # ...your steps, one comment per step...
    return d

# prices = clean_prices(prices_raw)

In [11]:
# TODO 2.6  Quality gates: assertions that must pass on the cleaned data
# assert prices["record_id"].is_unique

### 📒 Decision log

| Problem | Decision | Why |
|---|---|---|
| | | |

### 🧾 So what?: *(how much data did you lose, repair, or impute? how much does that matter?)*

✍️ 

---
# 3 · Descriptive statistics  *(10 pts)*

**Goal:** summarise centre, spread and shape, and choose the **right** statistic.

**Cover at least:**
* Price summary **by crop** (count, mean, median, std, min, max, **CV**).
* Prices **by year** (pivot crop × year) and the overall change 2021 → 2024.
* **Retail vs wholesale** (merge market type from `markets`), which is dearer, by how much, for which crops?
* Volume: totals and typical weekly volume by crop; skewness of price and volume.
* One **county ranking** for one crop (e.g. cheapest/dearest maize).

<details><summary>💡 Hint</summary>
`groupby().agg(...)` with named aggregation; `pivot_table`; `merge(markets[["market","type"]], on="market", validate="many_to_one")`; `.skew()`.
Remember: mean vs median for skewed data; CV (std/mean) to compare volatility across crops with different price levels.
</details>

In [12]:
# TODO 3.1  Summary by crop

In [13]:
# TODO 3.2  By year (pivot) and total change 2021 → 2024

In [14]:
# TODO 3.3  Retail vs wholesale

In [15]:
# TODO 3.4  Volume and skewness; a county ranking for one crop

### 🧾 So what?

✍️ 

---
# 4 · Visualisations  *(20 pts)*

**Goal:** at least **six** charts that each answer a *question* and have a *finding as title*.

**Required questions** (add your own!):
1. How are prices **distributed** for each crop? *(histogram/KDE, facets)*
2. How have prices **changed over time**, and what happened during the 2021–22 drought? *(lines; consider an index with 2021 = 100)*
3. Is there **seasonality**? In which months is each crop dearest? *(heatmap of detrended prices)*
4. How do **markets/counties** compare for one crop? *(box plot)*
5. How do price and **volume** relate: within a crop? *(scatter, facets)*
6. Where is **production** concentrated, and does it relate to price? *(county × crop heatmap or bars)*

**Chart standards:** title = finding · axis labels with units · bars from zero · sorted logically · no 3-D / rainbow · colour used with meaning.

<details><summary>💡 Hint: picking chart types</summary>
Distribution → histogram / KDE / box · Change over time → line · Compare groups → bar (from zero) or box · Two numbers → scatter · Two categories + a number → heatmap of a pivot table.
Detrend seasonality by dividing each price by the mean of its own series *in that year* (Lesson 7, section 6).
</details>

In [16]:
# TODO 4.1  Distributions by crop

In [17]:
# TODO 4.2  Prices over time (index, drought shading)

In [18]:
# TODO 4.3  Seasonality heatmap

In [19]:
# TODO 4.4  Markets or counties compared (one crop)

In [20]:
# TODO 4.5  Price vs volume, faceted by crop

In [21]:
# TODO 4.6  Production by county and crop

### 🧾 So what?: *(summarise the 3 most important things your charts show)*

✍️ 

---
# 5 · Correlation analysis  *(15 pts)*

**Goal:** go beyond "r = 0.4". **Quantify, then challenge.**

**Required analyses**
1. **Price vs volume.** Compute the *pooled* correlation across all rows, then the correlation **within each (crop, market) series**. Do they agree? Why or why not?
2. **Price vs rainfall.** Merge county-month weather with monthly prices. Does *last month's* rain relate to price? Try **lags 0, 1, 2, 3 months**, per crop. Remove the county/season/trend effects first (e.g. compare each price with its own series-year average).
3. **Crop vs crop.** Correlation heatmap of price **levels** and of weekly **changes**. What changes, and why?
4. **Production vs price** (county level, one or more crops): does higher production go with lower price? What else could explain the pattern?

**Always ask:** *Is a trend faking this? Is a group hiding it? Is a third variable behind it? How many points is this based on?*

<details><summary>💡 Hint: building the weather panel</summary>

1. Monthly mean price per `(crop, county, year, month)` with `groupby`.
2. In `weather`, sort by `county, year, month` and create `rain_lag1 = groupby("county").rainfall_mm.shift(1)` (and lags 2, 3).
3. `merge(..., on=["county","year","month"], validate="many_to_one")`.
4. Convert prices to *relative price* (divide by the series-year mean) so inflation and county price levels don't dominate, then correlate with rain lags.
</details>

<details><summary>💡 Hint: within-series correlation</summary>
`prices.groupby(["crop","market"])[["price_kes_per_kg","volume_kg"]].apply(lambda g: g["price_kes_per_kg"].corr(g["volume_kg"]))`
</details>

In [22]:
# TODO 5.1  Price vs volume: pooled vs within-series

In [23]:
# TODO 5.2  Price vs rainfall with lags (per crop)

In [24]:
# TODO 5.3  Crop-crop correlation: levels vs changes

In [25]:
# TODO 5.4  Production vs price (county level)

### 🧾 So what?: *(which relationships are strong, which are artefacts, and what would you test next?)*

✍️ 

---
# 6 · Observations, hypotheses & conclusions  *(15 pts)*

**Goal:** answer the four questions of the brief, with evidence.

1. **At least five findings**, each with a **number from your code** (use f-strings and `display(Markdown(...))`).
2. A **hypothesis table**:

| Observation (with number) | Hypothesis | How I would test it | What it means for ML |
|---|---|---|---|
| | | | |

3. **Limitations:** what *can't* this data tell us? (synthetic; 4 years; only 10 markets; weekly grain; unobserved drivers such as transport costs, imports, policy …)

<details><summary>💡 Hint: what makes a finding good?</summary>
*Specific* ("maize rose 29% in 2022"), *quantified*, *compared* (vs what?), *qualified* (how sure? how many points?), and *useful* (so what?).
</details>

In [26]:
# TODO 6.1  Key numbers, computed in code and printed as readable sentences
# display(Markdown(f"**Finding 1.** …"))

### Hypothesis table

| Observation | Hypothesis | Test | ML implication |
|---|---|---|---|
| | | | |

### Limitations

✍️ 

### Answers to the brief

1. *What's in the data and can we trust it?* ✍️
2. *What patterns?* ✍️
3. *What seems to drive prices, and what only looks like it?* ✍️
4. *Is it ready for ML?* (next section) ✍️

---
# 7 · ML-readiness  *(10 pts)*

**Goal:** show you know what a model would need.

**Cover**
1. **Candidate targets** (e.g. next-week price of a crop in a market; a "price spike" label; county production).
2. **Feature ideas**: group them: *lags & rolling stats · calendar (month, cyclical encoding) · weather (lagged rain) · market (type, county) · supply (volume lag)*. Build a few and show the result.
3. **Leakage:** list features that would be unknown at prediction time, or computed from the target.
4. **Train/test split:** why a *random* split is wrong for time series; do a **chronological** split (e.g. train 2021–2023, test 2024).
5. **Baseline:** compute a **naive forecast** (e.g. "same as last week") on the test period: report its error (MAE and %). *Any model must beat this.*
6. **Prep list:** encoding, scaling, transformations, missing-value policy.

<details><summary>💡 Hint: building lag features without leakage</summary>
Sort by `crop, market, date`; `groupby(["crop","market"])["price"].shift(1)` gives *last week's* price on this week's row: never use `shift(-1)` (that's tomorrow!). 
For rolling averages, `shift(1)` first, then `rolling(4).mean()`, so the window excludes the current row.
</details>

In [27]:
# TODO 7.1  Targets and features

In [28]:
# TODO 7.2  Chronological split + naive baseline error

### ML-readiness checklist

| Check | Status | Note |
|---|---|---|
| Target defined | | |
| Leakage removed | | |
| Missing values policy | | |
| Chronological split | | |
| Baseline to beat | | |
| Encoding / scaling plan | | |

### 🧾 So what?

✍️ 

---
# ✅ Before you submit: self-assessment

Tick each honestly (copy to a markdown cell and replace `[ ]` with `[x]`):

- [ ] The notebook runs **top to bottom** after *Restart & Run All*, no errors.
- [ ] Every issue in section 2 has **count + evidence + decision**, and quality gates pass.
- [ ] I used the **raw** prices file and my own cleaning, not a ready-made clean file.
- [ ] Every chart has a **finding title**, labelled axes with units, and sensible scales.
- [ ] My correlations are **challenged** (stratified, detrended, lagged, or explained).
- [ ] Every number in my conclusions was **computed in code**.
- [ ] Each section ends with a short **So what?**
- [ ] I stated **limitations** honestly.
- [ ] A non-programmer could follow the story in 20 minutes.

## 🚀 Stretch goals (optional, impressive)

* **Volatility league:** rank crop–market series by coefficient of variation; map volatility vs market size.
* **Price transmission:** do price changes at Wakulima lead those at Kongowea by a week or two? (cross-correlation of changes)
* **Drought impact estimate:** compare actual 2022–23 prices with a counterfactual built from 2021 + inflation; give a KES/kg and % estimate by crop.
* **Anomaly detector:** a rolling-median rule that flags weeks as *spikes*; evaluate how many it finds vs the planted typos you cleaned.
* **A one-page dashboard** (a single figure with 4–6 panels) suitable for a manager.

## 📬 Reference solution

When you have finished (or run out of time), open **`solutions/09_Project_Reference_Solution.ipynb`** and compare *approaches*: not just answers.
Where did you find something I missed? Where did I find something you didn't? That comparison is where the next level of skill comes from.

---
**Congratulations on completing Phase 1.** You can now explore an unfamiliar dataset, judge its quality, find its patterns, and say what a model could learn from it. 
Phase 2 builds on exactly this foundation. 🌱